# RS06 · Hierarchical state machines, fault latching and safe stopping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS06_hierarchical_state_machines_fault_latching_and_s.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 60 min

**Skill Hub concepts:** Hierarchical state machine; Fail-safe design and fault latching; Emergency stop categories; Compute partitioning and safety architecture

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS06")

## Map: a robot's top-level behavior as nested states

```
 ROOT ── handles "fault" from anywhere ──────────────────────────────> Fault  (latched: only "reset" leaves,
  ├── Init ──init_done──> Idle                                                  and only if the e-stop is released)
  └── Ready
       ├── Idle ──enable──> Manual
       └── Operational ── handles "disable" for both children ──> Idle
             ├── Manual ──auto──> Auto
             └── Auto ──manual──> Manual
```

An event is offered to the current state, then to its parent, and so on (unhandled events are ignored).
A transition exits states up to the **lowest common ancestor** (LCA) of source and target, then enters down to the
target. Behavior shared by children lives once in the parent: that is the point of hierarchy.

**Fail-safe design:** losing power, a wire or a heartbeat must lead to the safe state (brakes are spring-applied,
enable signals are active-high, safe torque off needs a signal present to allow torque). **Fault latching:** after a
fault the system stays in Fault until a deliberate reset, even if the cause vanished.

| Stop category (IEC 60204-1) | What happens | Typical use |
|---|---|---|
| 0 | power to the actuators removed immediately (uncontrolled stop) | simplest, but the axis coasts |
| 1 | controlled deceleration, **then** power removed (safe torque off) | most e-stops on servo robots |
| 2 | controlled deceleration, power stays on, position held | protective stop, not an e-stop |

**Safety architecture / compute partitioning:** the safety functions (e-stop chain, speed and position monitoring,
safe torque off) run on a separate, simpler, often dual-channel safety controller or a safety MCU core; the
application processor (ROS 2, planning, vision) is treated as untrusted and can only request motion.

## Task 1: transitions through the hierarchy
`transition` must append `"exit:<state>"` for every state left (innermost first) and `"entry:<state>"` for every state
entered (outermost first), then set `hsm.state`. `dispatch` offers the event up the parent chain; a handler returns a
target state or `None` (not handled). A target equal to the current state means "handled, stay".

In [ ]:
PARENT = {"Init": "ROOT", "Ready": "ROOT", "Fault": "ROOT", "Idle": "Ready", "Operational": "Ready",
          "Manual": "Operational", "Auto": "Operational", "ROOT": None}

class Hsm:
    def __init__(self):
        self.state, self.log, self.estop_released = "Init", [], False

HANDLERS = {
    ("Init", "init_done"): lambda h: "Idle",
    ("Idle", "enable"): lambda h: "Manual",
    ("Manual", "auto"): lambda h: "Auto",
    ("Auto", "manual"): lambda h: "Manual",
    ("Operational", "disable"): lambda h: "Idle",
    ("ROOT", "fault"): lambda h: "Fault",
    ("Fault", "fault"): lambda h: "Fault",
    ("Fault", "reset"): lambda h: "Idle" if h.estop_released else "Fault",
}

def ancestors(s):
    out = []
    while s is not None:
        out.append(s)
        s = PARENT[s]
    return out          # s, parent, ..., ROOT

def transition(hsm, target):
    raise NotImplementedError  # TODO

def dispatch(hsm, event):
    raise NotImplementedError  # TODO

lab.check("hsm", Hsm, transition, dispatch)

In [ ]:
%%writefile rs06_hsm.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/rs06.h
// Grade it with lab.check("hsm", "rs06_hsm.c"). For C++, name the file .cpp in both places.
#include "rs06.h"

void hsm_transition(Hsm *h, int target) {
    (void)h; (void)target; /* TODO: exits up to the least common ancestor, entries down to the target */
}

bool hsm_dispatch(Hsm *h, int event) {
    (void)h; (void)event;
    return false; /* TODO: try hsm_handler on the state, then each ancestor */
}

In [ ]:
# lab.check("hsm", "rs06_hsm.c")  # uncomment after writing the C/C++ cell above

## Task 2: pick the stop category

In [ ]:
stops = {
    "cut motor power at once through the contactor; the arm coasts": None,
    "ramp the joints down, then trigger safe torque off": None,
    "ramp down and hold position with the drives still powered (protective pause)": None,
    "e-stop on a heavy vertical axis: controlled stop, engage the brake, then remove torque": None,
}
lab.check("categories", stops)

## Checkpoint: Math
**M1 (Apply).** Category 1 stop from 1.5 rad/s at 3 rad/s$^2$. Seconds before power is removed?

**M2 (Apply).** Joint angle traveled during that stop, in radians?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** The Linux computer running ROS 2 also implements the e-stop in software. Best criticism?
- a) Linux is too slow to read a button
- b) safety functions should not depend on a complex, uncertified stack: a separate safety controller (or safety MCU) should own the e-stop and safe torque off
- c) ROS 2 has no e-stop message

**K2 (Analyze).** Why is "fault" handled in ROOT and "disable" in Operational instead of in each leaf state? What would
go wrong in a flat state machine as the robot grows?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs06.h`).
3. In `roboprep/checks/rs06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()